In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 

from scipy.signal import savgol_filter

from astropy.io import fits

workpath = '/data2/peng'
night='/2022-12-31'

# 1. Combine the spectra from two nodding positions

## 1.1 Load the data                                                                                         

In [ ]:
extracted_spectra_A = np.load(workpath+ '/' + night+'/extracted_spectra_position_A_sigmaclipper_0417.npy')
extracted_spectra_B = np.load(workpath+ '/' + night+'/extracted_spectra_position_B_sigmaclipper_0417.npy')

extracted_spectra_A_err = np.load(workpath+ '/' + night+'/extracted_spectra_position_A_err_0417.npy')
extracted_spectra_B_err = np.load(workpath+ '/' + night+'/extracted_spectra_position_B_err_0417.npy')
telluric_hdu = fits.open(f'{workpath}{night}/chi_tau/{night}/out/molecfit/TELLURIC_DATA.fits')
telluric_template = np.array(telluric_hdu[1].data)

#First reshape the telluric template to (7, 3, 2048)
telluric_template = np.reshape(telluric_template, (7, 3, 2048))  # (7, 3, 2048)
#organize the shape from the last order to the first order
telluric_template = np.array([telluric_template[i] for i in range(0,7)][::-1])  # (7, 3, 2048)

#Second, convert the shape to (3, 7, 2048) to match the extracted spectra shape
telluric_template = np.transpose(telluric_template, (1,0,2))  # (3, 7, 2048)


wave_hdu = fits.open(workpath + night +'/cal/WLEN_K2166_V_DH_Tau_A+B_center.fits')
wave_data = np.array(wave_hdu[1].data)[:,0:5,]  # (3, 5, 2048)




#plot the extracted spectra for each order and overplot the telluric template

fig, axs = plt.subplots(5,1, figsize=(20,15), sharex=False)

for order in range(5):
    index_plt = 4 - order

    axs[index_plt].plot(wave_data[:,order,:].flatten(), extracted_spectra_A[:,order,:].flatten(), label='Position A', color='blue', alpha=0.5, linewidth=0.6)
    axs[index_plt].plot(wave_data[:,order,:].flatten(), extracted_spectra_B[:,order,:].flatten(), label='Position B', color='orange', alpha=0.5, linewidth=0.6)

    axs[index_plt].plot(telluric_template[:,order,:]['lambda'].flatten()*1e3, telluric_template[:,order,:]['mtrans'].flatten()*60, label='Telluric Template', color='green', alpha=0.7, linewidth=1)

    #mark the regions with strong telluric absorption (mtrans < 0.7)
    strong_telluric = telluric_template[:,order,:]['mtrans'].flatten() < 0.7
    axs[index_plt].fill_between(telluric_template[:,order,:]['lambda'].flatten()*1e3, 0, 1, where=strong_telluric, color='gray', alpha=0.3, transform=axs[index_plt].get_xaxis_transform(), label='Strong Telluric Absorption (mtrans < 0.7)')

    axs[index_plt].set_ylabel('Flux (e-/s)')
    axs[index_plt].set_title(f'Order {order+1}')
    axs[index_plt].legend()


axs[4].set_xlabel('Wavelength (nm)')
plt.tight_layout()
plt.show()
# ── Airmass rescaling of the telluric template ──────────────────────────────
# The FITS header of SPEC_chiTau_PRIMARY.fits records only the last frame's
# airmass (1.568). The combined spectrum spans 6 equal-exposure frames at
# airmasses [1.819, 1.808, 1.799, 1.789, 1.569, 1.568], giving a true
# effective airmass of ≈1.725.  DH Tau B was observed at a mean airmass of
# ≈1.700 (night 2022-12-31) or ≈1.695 (night 2023-01-01).
#
# Since Chi Tau's template is slightly TOO DEEP relative to DH Tau B,
# dividing by it over-corrects → emission bumps at telluric positions.
#
# Fix: rescale via the Beer–Lambert airmass power law:
#   T_target(λ) = T_ref(λ) ^ (X_target / X_ref)   with  alpha = X_target/X_ref < 1
# This makes the template slightly shallower before masking and correction.
import os as _os, glob as _glob

_chi_raw = _os.path.join(workpath, night.lstrip('/'), 'chi_tau', night.lstrip('/'), 'raw')
_chi_sci = sorted([f for f in _glob.glob(_os.path.join(_chi_raw, '*.fits'))
                   if 'OBS' in _os.path.basename(f)])
_chi_airm = [(fits.getheader(f)['ESO TEL AIRM START'] + fits.getheader(f)['ESO TEL AIRM END']) / 2
             for f in _chi_sci]

_dh_raw = _os.path.join(workpath, night.lstrip('/'), 'raw')
_dh_sci = sorted([f for f in _glob.glob(_os.path.join(_dh_raw, '*.fits'))
                  if 'OBS' in _os.path.basename(f)])
_dh_airm = [(fits.getheader(f)['ESO TEL AIRM START'] + fits.getheader(f)['ESO TEL AIRM END']) / 2
            for f in _dh_sci]

airmass_chitau = float(np.mean(_chi_airm))
airmass_dhtaub = float(np.mean(_dh_airm))
alpha_airmass  = airmass_dhtaub / airmass_chitau   # < 1 → shallower template

print(f"Chi Tau  effective airmass : {airmass_chitau:.4f}  "
      "(header value 1.568 only reflects the last 2 of 6 co-added frames)")
print(f"DH Tau B effective airmass : {airmass_dhtaub:.4f}")
print(f"Airmass scaling factor  α  : {alpha_airmass:.4f}  "
      "(< 1 → reduces over-correction / emission bumps)")

# Apply scaling; preserve zeros for fully opaque channels
# telluric_template['mtrans'] shape = (3, 7, 2048)
telluric_mtrans = np.where(telluric_template['mtrans'] > 0,
                            telluric_template['mtrans'] ** alpha_airmass,
                            0.0)  # shape (3, 7, 2048), dtype float

## 1.2 Mark the wavelengths with transmission < 0.8

In [ ]:
# --- Sentinel-pixel masking ---
# The sigma-clipped extraction files (.._sigmaclipper_0417.npy) have already
# processed excalibuhr's original 1e10-error sentinels: sigma-clipping reduces
# them to ~6e5–9.3e5 in the file units, all just below the old 1e6 threshold.
# The old condition also required flux==0.0 exactly, missing the ~10 % of
# sentinels whose flux was set to a tiny non-zero floating-point artifact.
# Result: ~269 bad pixels per order per position propagated through telluric
# correction and flux calibration, producing error spikes ~1e9 × the median.
#
# Fix: threshold on error alone (no flux==0 requirement), lowered to 0.1.
# Good pixels have max raw error ~0.013 (file units); 0.1 sits safely above
# that and well below the sentinel floor of ~6e5.
_SENTINEL_ERR = 10
for _f, _e in [(extracted_spectra_A, extracted_spectra_A_err),
               (extracted_spectra_B, extracted_spectra_B_err)]:
    _sentinel = _e > _SENTINEL_ERR
    _f[_sentinel] = np.nan
    _e[_sentinel] = np.nan

_n_sent_A = np.isnan(extracted_spectra_A[:, 0:5, :]).sum() - np.isnan(extracted_spectra_A[:, 0:5, :] * 0).sum()
print(f'Sentinel pixels NaN-ified: '
      f'A={np.sum(extracted_spectra_A_err[:,0:5,:] != extracted_spectra_A_err[:,0:5,:])}'
      f'  (threshold: err > {_SENTINEL_ERR}  in file units)')

# simpler count: how many finite err remain vs. started
_n_total = extracted_spectra_A_err[:, 0:5, :].size
_n_nan_A = np.isnan(extracted_spectra_A_err[:, 0:5, :]).sum()
_n_nan_B = np.isnan(extracted_spectra_B_err[:, 0:5, :]).sum()
print(f'NaN after sentinel mask: A={_n_nan_A}/{_n_total}  B={_n_nan_B}/{_n_total}')

# --- Telluric mask: set pixels with transmission < 0.8 to NaN ---
mask_A = np.ones_like(extracted_spectra_A, dtype=bool)
mask_B = np.ones_like(extracted_spectra_B, dtype=bool)

for order in range(5):
    telluric_mtrans_order = telluric_mtrans[:,order,:]
    strong_telluric = telluric_mtrans_order < 0.8

    mask_A[:,order,:] = ~strong_telluric
    mask_B[:,order,:] = ~strong_telluric

extracted_spectra_A_masked = np.where(mask_A, extracted_spectra_A, np.nan)
extracted_spectra_B_masked = np.where(mask_B, extracted_spectra_B, np.nan)
extracted_spectra_A_err_masked = np.where(mask_A, extracted_spectra_A_err, np.nan)
extracted_spectra_B_err_masked = np.where(mask_B, extracted_spectra_B_err, np.nan)

#again plot the extracted spectra for each order after masking telluric regions and telluric template following the previous plotting code
fig, axs = plt.subplots(5,1, figsize=(20,15), sharex=False)

for order in range(5):
    index_plt = 4 - order

    axs[index_plt].plot(wave_data[:,order,:].flatten(), extracted_spectra_A_masked[:,order,:].flatten(), label='Position A', color='blue', alpha=0.5, linewidth=0.6)
    axs[index_plt].plot(wave_data[:,order,:].flatten(), extracted_spectra_B_masked[:,order,:].flatten(), label='Position B', color='orange', alpha=0.5, linewidth=0.6)

    axs[index_plt].plot(telluric_template[:,order,:]['lambda'].flatten()*1e3, telluric_mtrans[:,order,:].flatten()*60, label='Telluric Template (airmass-scaled)', color='green', alpha=0.7, linewidth=1)

    #mark the regions with strong telluric absorption (mtrans < 0.8)
    strong_telluric = telluric_mtrans[:,order,:].flatten() < 0.8
    axs[index_plt].fill_between(telluric_template[:,order,:]['lambda'].flatten()*1e3, 0, 1, where=strong_telluric, color='gray', alpha=0.3, transform=axs[index_plt].get_xaxis_transform(), label='Strong Telluric Absorption (mtrans < 0.8)')

    axs[index_plt].set_ylabel('Flux (e-/s)')
    axs[index_plt].set_title(f'Order {order+1} (Masked)')
    axs[index_plt].legend()
axs[4].set_xlabel('Wavelength (nm)')
plt.tight_layout()
plt.show()

#save the masked extracted spectra and errors
np.save(workpath+ night+'/extracted_spectra_position_A_masked.npy', extracted_spectra_A_masked)
np.save(workpath+ night+'/extracted_spectra_position_B_masked.npy', extracted_spectra_B_masked)
np.save(workpath+ night+'/extracted_spectra_position_A_err_masked.npy', extracted_spectra_A_err_masked)
np.save(workpath+ night+'/extracted_spectra_position_B_err_masked.npy', extracted_spectra_B_err_masked)


In [ ]:
#correct the tellurics by dividing the extracted spectra by the telluric template for both positions
extracted_spectra_A_corrected = extracted_spectra_A_masked / telluric_mtrans[:,0:5,:]
extracted_spectra_B_corrected = extracted_spectra_B_masked / telluric_mtrans[:,0:5,:]

#again plot the extracted spectra for each order after telluric correction following the previous plotting code
fig, axs = plt.subplots(5,1, figsize=(20,15), sharex=False)
for order in range(5):
    index_plt = 4 - order

    axs[index_plt].plot(wave_data[:,order,:].flatten(), extracted_spectra_A_corrected[:,order,:].flatten(), label='Position A', color='blue', alpha=0.5, linewidth=0.6)
    axs[index_plt].plot(wave_data[:,order,:].flatten(), extracted_spectra_B_corrected[:,order,:].flatten(), label='Position B', color='orange', alpha=0.5, linewidth=0.6)

    axs[index_plt].plot(telluric_template[:,order,:]['lambda'].flatten()*1e3, telluric_mtrans[:,order,:].flatten(), label='Telluric Template (airmass-scaled)', color='green', alpha=0.7, linewidth=1)

    axs[index_plt].set_ylabel('Flux (e-/s)')
    axs[index_plt].set_title(f'Order {order+1} (Telluric Corrected)')
    axs[index_plt].legend()
axs[4].set_xlabel('Wavelength (nm)')
plt.tight_layout()
plt.show()

# 2. Absolute flux calibration

## 2.1 Run absolute flux calibration with SIFONI data

In [ ]:

# ============================================================
# Absolute Flux Calibration using SINFONI/SIFONI data
# ============================================================
#
# SINFONI coverage (non-zero): 1100 – 2458.5 nm
# CRIRES+ K2166 coverage:      2063.7 – 2472.4 nm
# Overlap:                     2063.7 – 2458.5 nm
#
# All 5 CRIRES+ orders fall within the overlap.
# Order 1 (2422.4–2472.4 nm) has partial SINFONI coverage:
#   SINFONI data available up to ~2458.5 nm (~68% of the order).
#   The missing red tail (2458.5–2472.4 nm) is handled by the
#   polynomial response fit extrapolating from the other orders.
#
# Strategy:
#   1. Interpolate SINFONI onto CRIRES+ wavelength grid per order
#      using cubic spline interpolation.
#   2. Compute a per-order scale factor using the ratio of
#      integrated fluxes (sum-based, not pixel-wise) to avoid
#      being confused by resolution-mismatch between SINFONI
#      (low-res, R~1500) and CRIRES+ (high-res, R~100000).
#   3. Fit a degree-2 polynomial to the 5 scale factors vs.
#      order-centre wavelength → smooth instrument response curve.
#   4. Apply the smooth response per pixel to A_corrected and
#      B_corrected before combination.
#   5. Propagate the scale to the error arrays.
#
from scipy.interpolate import interp1d

# ---- 1. Load SINFONI spectrum ----
sinfoni_hdu = fits.open(workpath + '/DHTaub_SINFONIspeclib_JHK.fits')
sinfoni_raw  = sinfoni_hdu[0].data           # shape (14001, 2)
sinfoni_wave = sinfoni_raw[:, 0] * 1e3       # µm → nm
sinfoni_flux = sinfoni_raw[:, 1]             # raw file units (NOT W m⁻² µm⁻¹; see correction below)

# Absolute photometric anchor: DH Tau B K_s=14.19 (Patience+2012 Table 2, 2MASS K_s).
# The SINFONI speclib file has no BUNIT header and its K-band median (~6.8e-24)
# is ~1.3e8× smaller than the expected W m⁻² µm⁻¹ for K_s=14.19.  Rescale to
# physical units using the 2MASS K_s Vega zero-point (Cohen+2003: 666.7 Jy at 2.159 µm).
_K_dhtaub      = 14.19          # Patience+2012 Table 2, 2MASS K_s (±0.02)
_F0_K_Vega     = 4.29e-10       # 2MASS K_s Vega zero-point (666.7 Jy at 2.159 µm), W m⁻² µm⁻¹
_F_K_expected  = _F0_K_Vega * 10**(-_K_dhtaub / 2.5)
_wl_K          = (sinfoni_raw[:, 0] >= 2.0) & (sinfoni_raw[:, 0] <= 2.5)  # K band in µm
_sif_K_median  = np.nanmedian(sinfoni_flux[_wl_K & (sinfoni_flux > 0)])
_sif_abs_corr  = _F_K_expected / _sif_K_median
sinfoni_flux   = sinfoni_flux * _sif_abs_corr
print(f'SINFONI absolute correction: {_sif_abs_corr:.4e}')
print(f'  DH Tau b K={_K_dhtaub}: expected {_F_K_expected:.4e} W m⁻² µm⁻¹')
print(f'  SINFONI file K-band median before correction: {_sif_K_median:.4e}')

# Keep only finite, positive flux points
sif_valid  = np.isfinite(sinfoni_flux) & (sinfoni_flux > 0)
sif_wave_v = sinfoni_wave[sif_valid]
sif_flux_v = sinfoni_flux[sif_valid]

print(f'SINFONI valid range : {sif_wave_v.min():.1f} – {sif_wave_v.max():.1f} nm')
print(f'CRIRES+ range       : {wave_data.min():.2f} – {wave_data.max():.2f} nm')
print(f'Overlap             : {max(sif_wave_v.min(), wave_data.min()):.1f} – '
      f'{min(sif_wave_v.max(), wave_data.max()):.1f} nm')

# Build SINFONI cubic-spline interpolator (NaN outside range)
sinfoni_interp = interp1d(sif_wave_v, sif_flux_v, kind='cubic',
                          bounds_error=False, fill_value=np.nan)

# ---- 2. Interpolate SINFONI onto CRIRES+ wavelength grid ----
sinfoni_on_crires = sinfoni_interp(wave_data)   # (3, 5, 2048) [W m⁻² µm⁻¹]

# ---- 3. Per-order scale factors via integrated-flux ratio ----
#
# Rather than a noisy pixel-wise ratio (SINFONI R~1500 vs CRIRES+ R~100000
# → spectral features at very different depths), we compare the integrated
# flux in each order.  We smooth the CRIRES+ corrected spectrum with a
# broad Savitzky-Golay filter to suppress unresolved lines before summing.
#
from scipy.signal import savgol_filter

scale_factors   = np.full(5, np.nan)
scale_wave_cen  = np.zeros(5)
sinfoni_pix_cov = np.zeros(5, dtype=int)  # how many pixels have SINFONI coverage

for order in range(5):
    w_ord   = wave_data[:, order, :]               # (3, 2048)
    sif_ord = sinfoni_on_crires[:, order, :]       # (3, 2048)
    A_ord   = extracted_spectra_A_corrected[:, order, :]
    B_ord   = extracted_spectra_B_corrected[:, order, :]

    # Mask: SINFONI has valid flux AND both CRIRES+ beams are finite & positive
    sif_mask    = np.isfinite(sif_ord)
    crires_mean = np.nanmean(np.array([A_ord, B_ord]), axis=0)  # (3, 2048)
    data_mask   = np.isfinite(crires_mean) & (crires_mean > 0)
    joint_mask  = sif_mask & data_mask

    sinfoni_pix_cov[order] = joint_mask.sum()
    scale_wave_cen[order]  = np.nanmedian(w_ord[joint_mask]) if joint_mask.any() else np.nanmedian(w_ord)

    if joint_mask.sum() < 20:
        print(f'Order {order+1}: insufficient overlap ({joint_mask.sum()} pix) – scale=NaN')
        continue

    # Smooth CRIRES+ mean over a ~200-pixel window to suppress stellar lines
    # before computing the integrated-flux ratio
    crires_smooth = np.full_like(crires_mean, np.nan)
    for det in range(crires_mean.shape[0]):
        row = crires_mean[det, :]
        fin = np.isfinite(row)
        if fin.sum() > 201:
            row_smooth = savgol_filter(np.where(fin, row, 0.0),
                                       window_length=201, polyorder=2)
            crires_smooth[det, :] = np.where(fin, row_smooth, np.nan)

    smooth_mask = joint_mask & np.isfinite(crires_smooth) & (crires_smooth > 0)

    if smooth_mask.sum() < 10:
        # Fallback: use raw ratio with sigma clipping
        ratio = sif_ord[joint_mask] / crires_mean[joint_mask]
        med   = np.nanmedian(ratio)
        std   = np.nanstd(ratio)
        keep  = np.abs(ratio - med) < 3 * std
        scale_factors[order] = np.nanmedian(ratio[keep]) if keep.any() else med
        print(f'Order {order+1} [fallback sigma-clip]: scale = {scale_factors[order]:.4e}  '
              f'(n={keep.sum()})')
    else:
        # Integrated-flux ratio: Σ SINFONI / Σ CRIRES+(smoothed)
        scale_factors[order] = (np.nansum(sif_ord[smooth_mask]) /
                                np.nansum(crires_smooth[smooth_mask]))
        print(f'Order {order+1}: scale = {scale_factors[order]:.4e}  '
              f'(n_pix={smooth_mask.sum()}, SINFONI coverage={sinfoni_pix_cov[order]} pix, '
              f'wave_cen={scale_wave_cen[order]:.1f} nm)')

# ---- 4. Polynomial response fit across all orders ----
#
# A degree-2 polynomial in wavelength captures the broad shape of
# the combined SED.
# It also naturally extrapolates the scale into Order 1's uncovered tail
# (2458.5–2472.4 nm) where SINFONI data ends.
#
valid_ord = np.isfinite(scale_factors)
print(f'\nFitting polynomial with {valid_ord.sum()} valid orders ...')

if valid_ord.sum() >= 3:
    poly_coeffs = np.polyfit(scale_wave_cen[valid_ord],
                             scale_factors[valid_ord], deg=2)
    poly_resp   = np.poly1d(poly_coeffs)
    print(f'Poly coefficients (degree-2): {poly_coeffs}')
    use_poly = True
elif valid_ord.sum() >= 1:
    use_poly = False
    fallback_scale = np.nanmedian(scale_factors)
    print(f'Too few valid orders – using median scale = {fallback_scale:.4e}')
else:
    raise RuntimeError('No valid scale factors found – check SINFONI/CRIRES+ overlap')

# ---- Diagnostic: plot scale factors and polynomial fit ----
fig_sc, ax_sc = plt.subplots(figsize=(7, 4))
ax_sc.scatter(scale_wave_cen[valid_ord], scale_factors[valid_ord],
              color='steelblue', zorder=5, label='Per-order scale')
if use_poly:
    w_fit = np.linspace(wave_data.min() - 20, wave_data.max() + 20, 300)
    ax_sc.plot(w_fit, poly_resp(w_fit), 'r--', label='Polynomial fit (deg 2)')
    ax_sc.axvline(sif_wave_v.max(), color='gray', linestyle=':', alpha=0.7,
                  label=f'SINFONI ends ({sif_wave_v.max():.1f} nm)')
ax_sc.set_xlabel('Wavelength (nm)')
ax_sc.set_ylabel('Scale factor  [W m⁻² µm⁻¹ / (e⁻ s⁻¹)]')
ax_sc.set_title('Instrument Response: SINFONI / CRIRES+ (per order)')
ax_sc.legend(fontsize=9)
ax_sc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(workpath + night + '/flux_cal_scale_factors.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- 5. Apply flux calibration to A and B corrected spectra ----
extracted_spectra_A_flux_cal = np.full_like(extracted_spectra_A_corrected, np.nan)
extracted_spectra_B_flux_cal = np.full_like(extracted_spectra_B_corrected, np.nan)

# Propagate errors through telluric correction + flux scaling
# Bug 2 fix: use airmass-scaled telluric_mtrans (same template as flux), not raw telluric_template['mtrans'].
# Bug 3 fix: use already-telluric-masked error arrays from Cell 5 (extracted_spectra_A_err_masked)
# so NaN-masked flux pixels are also NaN in the errors, preventing nansum from mixing
# masked and unmasked errors in the combining step.
extracted_spectra_A_err_corrected = extracted_spectra_A_err_masked / telluric_mtrans[:, 0:5, :]
extracted_spectra_B_err_corrected = extracted_spectra_B_err_masked / telluric_mtrans[:, 0:5, :]

extracted_spectra_A_err_flux_cal = np.full_like(extracted_spectra_A_err_corrected, np.nan)
extracted_spectra_B_err_flux_cal = np.full_like(extracted_spectra_B_err_corrected, np.nan)

for order in range(5):
    w_ord = wave_data[:, order, :]   # (3, 2048)

    if use_poly:
        scale_map = poly_resp(w_ord)  # smooth per-pixel scale
    else:
        scale_map = np.full_like(w_ord, fallback_scale)

    extracted_spectra_A_flux_cal[:, order, :]     = extracted_spectra_A_corrected[:, order, :]     * scale_map
    extracted_spectra_B_flux_cal[:, order, :]     = extracted_spectra_B_corrected[:, order, :]     * scale_map
    extracted_spectra_A_err_flux_cal[:, order, :] = extracted_spectra_A_err_corrected[:, order, :] * scale_map
    extracted_spectra_B_err_flux_cal[:, order, :] = extracted_spectra_B_err_corrected[:, order, :] * scale_map

# ---- 6. Diagnostic: compare flux-calibrated spectra against SINFONI ----
fig, axs = plt.subplots(5, 1, figsize=(20, 15), sharex=False)

for order in range(5):
    idx = 4 - order
    w   = wave_data[:, order, :].flatten()

    axs[idx].plot(w, extracted_spectra_A_flux_cal[:, order, :].flatten(),
                  label='A (flux-cal)', color='blue', alpha=0.5, linewidth=0.6)
    axs[idx].plot(w, extracted_spectra_B_flux_cal[:, order, :].flatten(),
                  label='B (flux-cal)', color='orange', alpha=0.5, linewidth=0.6)

    sif_sel = (sif_wave_v >= w.min() - 10) & (sif_wave_v <= w.max() + 10)
    axs[idx].plot(sif_wave_v[sif_sel], sif_flux_v[sif_sel],
                  label='SINFONI', color='red', alpha=0.6, linewidth=1)

    if order == 0:
        axs[idx].axvline(sif_wave_v.max(), color='gray', linestyle=':',
                         label=f'SINFONI ends ({sif_wave_v.max():.1f} nm) → poly extrapolation beyond here')

    axs[idx].set_ylabel('Flux [W m⁻² µm⁻¹]')
    axs[idx].set_title(f'Order {order+1} – Flux Calibrated  '
                       f'(scale={scale_factors[order]:.3e}  W m⁻² µm⁻¹ / e⁻ s⁻¹)')
    axs[idx].legend(fontsize=8)
    axs[idx].set_xlim(w.min() - 5, w.max() + 5)

    axs[idx].set_ylim(0, np.nanmax([extracted_spectra_A_flux_cal[:, order, :].max(),
                                    extracted_spectra_B_flux_cal[:, order, :].max(),
                                    sif_flux_v[sif_sel].max()]) * 3)

axs[4].set_xlabel('Wavelength (nm)')
plt.tight_layout()
plt.savefig(workpath + night + '/flux_cal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- 7. Save flux-calibrated spectra ----
np.save(workpath + '/' + night + '/extracted_spectra_position_A_flux_cal.npy',
        extracted_spectra_A_flux_cal)
np.save(workpath + '/' + night + '/extracted_spectra_position_B_flux_cal.npy',
        extracted_spectra_B_flux_cal)
np.save(workpath + '/' + night + '/extracted_spectra_position_A_err_flux_cal.npy',
        extracted_spectra_A_err_flux_cal)
np.save(workpath + '/' + night + '/extracted_spectra_position_B_err_flux_cal.npy',
        extracted_spectra_B_err_flux_cal)
print('Saved flux-calibrated spectra (A, B) and propagated errors.')


In [ ]:

# ---- Combine the two positions using flux-calibrated spectra ----
# Formal error propagation (LPU / GUM §5): for f = (A + B) / N,
# sigma_f = sqrt(sigma_A^2 + sigma_B^2) / N.
# For N=2, median == mean exactly, so the same formula applies.
# n_valid handles pixels where one nod is NaN: error is then just the single valid error.

extracted_spectra_combined = np.nanmedian(
    np.array([extracted_spectra_A_flux_cal, extracted_spectra_B_flux_cal]), axis=0)

_stack_fc     = np.array([extracted_spectra_A_flux_cal, extracted_spectra_B_flux_cal])
_err_stack_fc = np.array([extracted_spectra_A_err_flux_cal, extracted_spectra_B_err_flux_cal])
_n_valid_fc   = np.sum(np.isfinite(_stack_fc), axis=0)          # 0, 1, or 2 per pixel
extracted_spectra_combined_err = np.where(
    _n_valid_fc > 0,
    np.sqrt(np.nansum(_err_stack_fc**2, axis=0)) / _n_valid_fc,
    np.nan)

# Save the combined flux-calibrated spectra
np.save(workpath + '/' + night + '/extracted_spectra_combined_flux_cal.npy',
        extracted_spectra_combined)
np.save(workpath + '/' + night + '/extracted_spectra_combined_err_flux_cal.npy',
        extracted_spectra_combined_err)

# Also save the non-flux-calibrated combined spectra (kept for reference)
extracted_spectra_combined_raw = np.nanmedian(
    np.array([extracted_spectra_A_corrected, extracted_spectra_B_corrected]), axis=0)

_stack_raw     = np.array([extracted_spectra_A_corrected, extracted_spectra_B_corrected])
_err_stack_raw = np.array([extracted_spectra_A_err_corrected, extracted_spectra_B_err_corrected])
_n_valid_raw   = np.sum(np.isfinite(_stack_raw), axis=0)
extracted_spectra_combined_raw_err = np.where(
    _n_valid_raw > 0,
    np.sqrt(np.nansum(_err_stack_raw**2, axis=0)) / _n_valid_raw,
    np.nan)

np.save(workpath + '/' + night + '/extracted_spectra_combined_sigmaclipper.npy',
        extracted_spectra_combined_raw)
np.save(workpath + '/' + night + '/extracted_spectra_combined_err_sigmaclipper.npy',
        extracted_spectra_combined_raw_err)

print('Combined spectra shape:', extracted_spectra_combined.shape)
print('Units: W m⁻² µm⁻¹  (flux-calibrated via SINFONI)')


In [ ]:
# ---- Summary plot: absolute flux calibration with photometric anchor ----

_K_eff_nm    = 2159.0
_K_bw_nm     = 131.0
_K_mag_err   = 0.02
_F_anchor    = _F_K_expected
_F_anchor_err = _F_anchor * np.log(10) / 2.5 * _K_mag_err

# Flatten and sort by wavelength — required for fill_between and plot to render
# correctly; raw flatten() interleaves detectors/orders in array order, not λ order.
crires_wave_all = wave_data[:, 0:5, :].flatten()
crires_flux_all = extracted_spectra_combined.flatten()
crires_err_all  = extracted_spectra_combined_err.flatten()

_sort = np.argsort(crires_wave_all)
crires_wave_all = crires_wave_all[_sort]
crires_flux_all = crires_flux_all[_sort]
crires_err_all  = crires_err_all[_sort]

fig, ax = plt.subplots(figsize=(13, 5))

# 1σ error band as gray fill, flux as steelblue curve
ax.fill_between(crires_wave_all,
                crires_flux_all - crires_err_all,
                crires_flux_all + crires_err_all,
                color='gray', alpha=0.3, linewidth=1, zorder=2)
ax.plot(crires_wave_all, crires_flux_all,
        color='steelblue', alpha=0.85, linewidth=0.7, zorder=3,
        label='CRIRES+ combined (flux-cal, R∼100000)')

# SINFONI low-res spectrum
ax.plot(sif_wave_v, sif_flux_v,
        color='tomato', alpha=0.9, linewidth=1.5, zorder=4,
        label='SINFONI (scaled to K$_s$ anchor, R∼1500)')

# 2MASS K_s photometric anchor
ax.errorbar(_K_eff_nm, _F_anchor,
            xerr=_K_bw_nm, yerr=_F_anchor_err,
            fmt='*', color='gold', markeredgecolor='k', markeredgewidth=0.8,
            markersize=16, capsize=5, capthick=1.5, linewidth=1.5, zorder=10,
            label=(f'2MASS K$_s$ anchor  ($K_s = {_K_dhtaub}$ mag, Patience+2012)\n'
                   f'  $F_{{K_s}} = {_F_anchor:.2e}$ W m$^{{-2}}$ µm$^{{-1}}$'))

ax.set_xlim(2000, 2520)

# Y upper limit: 99.5th percentile of finite CRIRES+ flux and SINFONI in the
# plot window, plus anchor + error — take the largest, add 30 % headroom.
_crires_fin = crires_flux_all[np.isfinite(crires_flux_all)]
_sif_in_win = sif_flux_v[(sif_wave_v >= 2000) & (sif_wave_v <= 2520)]
_ymax = max(
    np.nanpercentile(_crires_fin, 99.5) if len(_crires_fin) else 0,
    np.nanmax(_sif_in_win)              if len(_sif_in_win) else 0,
    _F_anchor + _F_anchor_err,
) * 1.8
ax.set_ylim(0, _ymax)

ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Flux  [W m$^{-2}$ µm$^{-1}$]', fontsize=12)
ax.set_title('DH Tau B – Absolute Flux Calibration Summary', fontsize=13)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(workpath + night + '/flux_cal_summary.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
#again plot the combined extracted spectra for each order following the previous plotting code
fig, axs = plt.subplots(5,3, figsize=(20,20), sharex=False)
for order in range(5):
    index_plt = 4 - order

    for det in range(3):
        
        axs[index_plt, det].plot(wave_data[det, order, :].flatten(),
                                 extracted_spectra_combined[det, order, :].flatten(),
                                 label='Combined Spectrum', color='purple', alpha=0.7, linewidth=0.6)
        
        '''
        axs[index_plt, det].errorbar(wave_data[det, order, :].flatten(),
                                     extracted_spectra_combined[det, order, :].flatten(),
                                     yerr=extracted_spectra_combined_err[det, order, :].flatten(),
                                     fmt='o', color='gray', alpha=0.5, markersize=0.1,
                                     label='Combined Spectrum Error')
        '''

        axs[index_plt, det].plot(telluric_template[det, order, :]['lambda'].flatten() * 1e3,
                                 telluric_mtrans[det, order, :].flatten(),
                                 label='Telluric Template (airmass-scaled)', color='green', alpha=0.7, linewidth=1)

        axs[index_plt, det].set_ylabel('Flux (W m⁻² µm⁻¹)')
        axs[index_plt, det].set_title(f'Order {order+1} (Combined Spectrum) – Detector {det+1}')
        axs[index_plt, det].legend()

        axs[index_plt, det].set_ylim(-1e-15, 4e-15)
    
axs[4, 0].set_xlabel('Wavelength (nm)')
plt.tight_layout()
plt.show()

## 2.2 Run SNR check 

In [ ]:
#Calculate the SNR of the combined spectrum for each order and plot it
extracted_spectra_combined = np.load(workpath + '/' + night + '/extracted_spectra_combined_sigmaclipper.npy')
extracted_spectra_combined_err = np.load(workpath + '/' + night + '/extracted_spectra_combined_err_sigmaclipper.npy')

wave_hdu = fits.open(workpath + night + '/cal/WLEN_K2166_V_DH_Tau_A+B_center.fits')
wave_data = np.array(wave_hdu[1].data)[:, 0:5, ]  # (3, 5, 2048)

fig, axs = plt.subplots(5, 1, figsize=(20, 20), sharex=False)
for order in range(5):
    index_plt = 4 - order

    # Formal per-pixel SNR from propagated error array
    snr = extracted_spectra_combined[:, order, :].flatten() / extracted_spectra_combined_err[:, order, :].flatten()

    axs[index_plt].plot(wave_data[:,order,:].flatten(), snr, label='SNR of Combined Spectrum', color='purple', alpha=0.7, linewidth=0.6)

    axs[index_plt].set_ylabel('SNR')
    axs[index_plt].set_title(f'Order {order+1} (Combined Spectrum SNR)')
    axs[index_plt].legend()

    axs[index_plt].set_ylim(0, 10)

plt.tight_layout()
plt.show()

#Calulate the SNR for each nodding position and plot it (empirical — error arrays not in scope here)
extracted_spectra_A = np.load(workpath+ '/' + night+'/extracted_spectra_position_A_sigmaclipper_0417.npy')
extracted_spectra_B = np.load(workpath+ '/' + night+'/extracted_spectra_position_B_sigmaclipper_0417.npy')

extracted_spectra_A_err = np.load(workpath + '/' + night+'/extracted_spectra_position_A_err_0417.npy')
extracted_spectra_B_err = np.load(workpath + '/' + night+'/extracted_spectra_position_B_err_0417.npy')

fig, axs = plt.subplots(5, 1, figsize=(20, 20), sharex=False)
for order in range(5):
    index_plt = 4 - order

    snr_A = extracted_spectra_A[:,order,:].flatten() / extracted_spectra_A_err[:,order,:].flatten()
    snr_B = extracted_spectra_B[:,order,:].flatten() / extracted_spectra_B_err[:,order,:].flatten()

    axs[index_plt].plot(wave_data[:,order,:].flatten(), snr_A, label='SNR of Position A', color='blue', alpha=0.7, linewidth=0.6)
    axs[index_plt].plot(wave_data[:,order,:].flatten(), snr_B, label='SNR of Position B', color='orange', alpha=0.7, linewidth=0.6)

    axs[index_plt].set_ylabel('SNR')
    axs[index_plt].set_title(f'Order {order+1} (Position A and B SNR)')
    axs[index_plt].legend()

    axs[index_plt].set_ylim(0, 10)

plt.tight_layout()  
plt.show()

#Calulate the SNR for the combined spectra before flux calibration and plot it
extracted_spectra_combined_raw = np.load(workpath + '/' + night + '/extracted_spectra_combined_sigmaclipper.npy')
extracted_spectra_combined_raw_err = np.load(workpath + '/' + night + '/extracted_spectra_combined_err_sigmaclipper.npy')

fig, axs = plt.subplots(5, 1, figsize=(20, 20), sharex=False)
for order in range(5):
    index_plt = 4 - order

    # Formal per-pixel SNR from propagated error array
    snr_combined_raw = extracted_spectra_combined_raw[:, order, :].flatten() / extracted_spectra_combined_raw_err[:, order, :].flatten()

    axs[index_plt].plot(wave_data[:,order,:].flatten(), snr_combined_raw, label='SNR of Combined Spectrum (pre-flux-cal)', color='purple', alpha=0.7, linewidth=0.6)

    axs[index_plt].set_ylabel('SNR')
    axs[index_plt].set_title(f'Order {order+1} (Combined Spectrum SNR pre-flux-cal)')
    axs[index_plt].legend()

    axs[index_plt].set_ylim(0, 10)

plt.tight_layout()
plt.show()


# 3. Run simple Cross Correlation test to obtain the observed RV and CC SNR 

In [ ]:
# ============================================================
# CO Cross-Correlation on Combined Spectra
# ============================================================
from scipy import interpolate

# ---- processor_cross_correlation class (from cooking_subtraction.ipynb Cell 4) ----
#load barycentric corrected wavelength grid
#wave_data = np.load(workpath +'/' + night + '/barycentric_wavelengths_night1.npy')  # shape (3, 5, 2048)

class processor_cross_correlation:
    def __init__(self, wMod, fMod, wlen, cube, nOrder, n_spatial, nDet):
        """
        wMod, fMod : 1D model wavelength and flux (same length)
        wlen : observed wavelength grid with shape (nDet, nOrder, nPix)
        cube : observed cube shape (nDet, nOrder, n_spatial, nPix)
        """
        self.wMod = wMod
        self.fMod = fMod
        self.wlen = wlen
        self.cube = cube
        self.n_spatial = n_spatial
        self.nOrder = nOrder
        self.nDet = nDet

    def xcorr(self, f, g):
        """
        Normalized cross-correlation between 1D arrays f and g.
        Returns NaN if variance is zero.
        """
        f = np.asarray(f, dtype=float)
        g = np.asarray(g, dtype=float)
        nx = len(f)
        if nx == 0:
            return np.nan
        I = np.ones(nx)
        f_mean_sub = f - (np.dot(f, I) / nx)
        g_mean_sub = g - (np.dot(g, I) / nx)
        R    = np.dot(f_mean_sub, g_mean_sub) / nx
        varf = np.dot(f_mean_sub, f_mean_sub) / nx
        varg = np.dot(g_mean_sub, g_mean_sub) / nx
        denom = np.sqrt(varf * varg)
        if denom == 0 or np.isnan(denom):
            return np.nan
        return R / denom

    def get_cc_grid(self, rvlag, ncc):
        """
        Compute CCF cube with shape (nDet, nOrder, n_spatial, ncc).
        """
        ccf = np.zeros((self.nDet, self.nOrder, self.n_spatial, ncc))
        coef_spline = interpolate.splrep(self.wMod, self.fMod, s=0.0)

        for irv, rv in enumerate(rvlag):
            beta   = rv / 2.998e5
            wShift = self.wlen * np.sqrt((1.0 - beta) / (1.0 + beta))
            intMod = interpolate.splev(wShift, coef_spline, der=0)  # (nDet, nOrder, nPix)

            for iDet in range(self.nDet):
                for iOrder in range(self.nOrder):
                    for iObs in range(self.n_spatial):
                        obs       = self.cube[iDet, iOrder, iObs, :]
                        model_row = intMod[iDet, iOrder, :]
                        mask = np.isfinite(obs) & np.isfinite(model_row)
                        if np.sum(mask) < 5:
                            ccf[iDet, iOrder, iObs, irv] = np.nan
                        else:
                            ccf[iDet, iOrder, iObs, irv] = self.xcorr(obs[mask], model_row[mask])

        self.rvlag = rvlag
        self.ncc   = ncc
        return ccf

    def ccf_tot(self, rvlag, ncc, plot=True, subtract_continuum='med_flux',
                normalization='median subtracted', v_sys=None, clean_grids=None,
                po=None, central_pix=None, spatial_pix=None):
        """
        Returns ccf_Sum and ccf_SNR, each shape (n_spatial, ncc),
        summed across detectors and orders.
        """
        medFlux = np.zeros((self.nDet, self.nOrder, self.wlen.shape[-1]))
        for iDet in range(self.nDet):
            for iOrder in range(self.nOrder):
                medFlux[iDet, iOrder, :] = np.nanmedian(self.cube[iDet, iOrder, :, :], axis=0)

        ccfWeight = np.sum(medFlux, axis=(0, 1))
        if np.nansum(ccfWeight) != 0:
            ccfWeight = ccfWeight / np.nansum(ccfWeight)

        ccf     = self.get_cc_grid(rvlag, ncc)      # (nDet, nOrder, n_spatial, ncc)
        ccf_Sum = np.nansum(ccf, axis=(0, 1))       # (n_spatial, ncc)

        if normalization == 'median':
            denom = np.nanmedian(ccf_Sum)
            if denom != 0:
                ccf_Sum = (ccf_Sum - denom) / denom
        elif normalization == 'max':
            mval = np.nanmax(ccf_Sum)
            if mval != 0:
                ccf_Sum = ccf_Sum / mval
        elif normalization == 'median subtracted':
            for iObs in range(self.n_spatial):
                med = np.nanmedian(ccf_Sum[iObs, :])
                ccf_Sum[iObs, :] -= med

        if plot:
            plt.figure(figsize=(10, 4))
            plt.imshow(ccf_Sum, aspect='auto', origin='lower',
                       extent=(rvlag[0], rvlag[-1], 0, self.n_spatial))
            plt.colorbar(label='CCF')
            plt.xlabel('RV lag (km/s)')
            plt.ylabel('Spatial index')
            plt.title('CCF map')
            if central_pix is not None:
                plt.hlines(y=central_pix, xmin=rvlag[0], xmax=rvlag[-1],
                           ls='--', colors='lightgray')
            if v_sys is not None:
                plt.vlines(x=v_sys, ymin=0, ymax=(self.n_spatial - 1),
                           ls='--', colors='lightgray')
            plt.tight_layout()
            plt.show()

        if clean_grids is None:
            std_ccf = np.nanstd(ccf_Sum)
        else:
            g0, g1 = int(clean_grids[0][0]), int(clean_grids[0][1])
            g2, g3 = int(clean_grids[1][0]), int(clean_grids[1][1])
            ccf_clean_map = np.concatenate((ccf_Sum[:, g0:g1], ccf_Sum[:, g2:g3]), axis=1)
            std_ccf = np.nanstd(ccf_clean_map)

        ccf_SNR = np.full_like(ccf_Sum, np.nan)
        if std_ccf != 0:
            ccf_SNR = ccf_Sum / std_ccf

        if plot:
            plt.figure(figsize=(10, 4))
            plt.imshow(ccf_SNR, aspect='auto', origin='lower',
                       extent=(rvlag[0], rvlag[-1], 0, self.n_spatial))
            plt.colorbar(label='SNR')
            plt.xlabel('RV lag (km/s)')
            plt.ylabel('Spatial index')
            plt.title('CCF SNR map')
            if central_pix is not None:
                plt.hlines(y=central_pix, xmin=rvlag[0], xmax=rvlag[-1],
                           ls='--', colors='lightgray')
            if v_sys is not None:
                plt.vlines(x=v_sys, ymin=0, ymax=(self.n_spatial - 1),
                           ls='--', colors='lightgray')
            plt.tight_layout()
            plt.show()

        return (ccf_Sum, ccf_SNR)


# ---- Load CO template spectrum ----
# File layout: shape (2, N) where row 0 = wavelength [m], row 1 = flux
co_template_path = workpath + '/spectra_input/planet_2300_4.0_-0.0_0_co_only.dat'
co_template  = np.loadtxt(co_template_path)
ccf_model_w  = co_template[0]   # wavelength in metres
ccf_model_f  = co_template[1]   # flux (arbitrary units)

# Sort ascending in wavelength (required by splrep)
sort_idx    = np.argsort(ccf_model_w)
ccf_model_w = ccf_model_w[sort_idx]
ccf_model_f = ccf_model_f[sort_idx]

print(f'CO template: {len(ccf_model_w)} points, '
      f'wavelength {ccf_model_w.min()*1e9:.1f} - {ccf_model_w.max()*1e9:.1f} nm')

# ---- RV lag grid: -100 to +100 km/s, step 1 km/s ----
RVlag = np.arange(-100, 101, 1)   # 201 points
ncc   = len(RVlag)

# ---- Prepare the data cube ----
# extracted_spectra_combined shape: (nDet=3, nOrder=5, nPix=2048)
# Add singleton spatial axis -> (nDet, nOrder, n_spatial=1, nPix)
nDet, nOrder, nPix = extracted_spectra_combined.shape
cube_combined = extracted_spectra_combined[:, :, np.newaxis, :]  # (3, 5, 1, 2048)

print(f'Obs cube shape  : {cube_combined.shape}')
print(f'Wave grid shape : {wave_data.shape}')
print(f'RV lag          : {RVlag[0]} to {RVlag[-1]} km/s  ({ncc} steps)')

# ---- Instantiate cross-correlator ----
process_cc_combined = processor_cross_correlation(
    wMod      = ccf_model_w * 1e9,   # metres -> nm to match wave_data
    fMod      = ccf_model_f,
    wlen      = wave_data,            # (nDet, nOrder, nPix) in nm
    cube      = cube_combined,        # (nDet, nOrder, 1, nPix)
    nOrder    = nOrder,
    nDet      = nDet,
    n_spatial = 1
)

# ---- Run cross-correlation ----
ccf_sum, ccf_snr = process_cc_combined.ccf_tot(
    rvlag         = RVlag,
    ncc           = ncc,
    plot          = False,
    normalization = 'median subtracted',
    clean_grids = [(0, 50), (150, 200)]
)

# ccf_sum / ccf_snr shape (1, 201) -> squeeze to 1D
ccf_1d     = ccf_sum[0, :]
ccf_snr_1d = ccf_snr[0, :]

peak_rv  = RVlag[np.nanargmax(ccf_snr_1d)]
peak_snr = np.nanmax(ccf_snr_1d)
print(f'\nPeak CCF SNR = {peak_snr:.2f}  at  RV = {peak_rv:.1f} km/s')


#---- Run auto corss-correlation -----

#using spline interpolation and reshape the model flux to (nDet, nOrder, 1, nPix) to match the cube shape
ccf_model_f_int = np.zeros((nDet, nOrder, nPix))
coef_spline = interpolate.splrep(ccf_model_w * 1e9, ccf_model_f, s=0.0)  # spline coefficients for the model
splev = interpolate.splev  # ensure local name used below is defined
for iDet in range(nDet):
    for iOrder in range(nOrder):
        ccf_model_f_int[iDet, iOrder, :] = splev(wave_data[iDet, iOrder, :], coef_spline, der=0)
        


ccf_model_f_reshaped = ccf_model_f_int[:, :, np.newaxis, :]  # (nDet, nOrder, 1, nPix)

process_cc_auto = processor_cross_correlation(
    wMod      = ccf_model_w * 1e9,   # metres -> nm to match wave_data
    fMod      = ccf_model_f,
    wlen      = wave_data,            # (nDet, nOrder, nPix) in nm
    cube      = ccf_model_f_reshaped,                  # (nDet, nOrder, 1, nPix)
    nOrder    = nOrder,
    nDet      = nDet,
    n_spatial = 1
)

ccf_auto, ccf_auto_snr = process_cc_auto.ccf_tot(
    rvlag         = RVlag,  
    ncc           = ncc,
    plot          = False,
    normalization = 'median subtracted',
    clean_grids = [(0, 50), (150, 200)]

)

ccf_auto_1d = ccf_auto[0, :]
ccf_auto_snr_1d = ccf_auto_snr[0, :]

peak_rv_auto = RVlag[np.nanargmax(ccf_auto_snr_1d)]
peak_snr_auto = np.nanmax(ccf_auto_snr_1d)
print(f'\nAuto CCF Peak SNR = {peak_snr_auto:.2f}  at  RV = {peak_rv_auto:.1f} km/s')

# ---- Plot results ----
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(RVlag, ccf_1d, color='steelblue', linewidth=1.5, label='CO CCF')
axes[0].axvline(0,       color='gray',   linestyle='--', linewidth=0.8, label='RV = 0')
axes[0].axvline(peak_rv, color='tomato', linestyle='--', linewidth=1.0,
                label=f'Peak at {peak_rv:.0f} km/s')
#axes[0].plot(RVlag, ccf_auto_1d, color='darkorange', linewidth=1.5, label='Auto CCF')


axes[0].set_ylabel('CCF (median-subtracted)')
axes[0].set_title('CO Cross-Correlation - Combined Spectra\n'
                  '(template: T=2300 K, log g=4.0, [M/H]=0.0, CO only)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(RVlag, ccf_snr_1d, color='darkorange', linewidth=1.5, label='CO CCF SNR')
axes[1].axvline(0,       color='gray',   linestyle='--', linewidth=0.8)
axes[1].axvline(peak_rv, color='tomato', linestyle='--', linewidth=1.0,
                label=f'Peak SNR = {peak_snr:.2f}')
axes[1].set_xlabel('RV lag (km/s)')
axes[1].set_ylabel('SNR')
axes[1].set_title('CO CCF SNR')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(workpath + night + '/co_ccf_combined_spectra.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
#----- Load H2O template spectrum ----
h2o_template_path = workpath + '/spectra_input/planet_2300_4.0_-0.0_0_h2o_pokazatel_only.dat'
h2o_template = np.loadtxt(h2o_template_path)
ccf_model_w = h2o_template[0]
ccf_model_f = h2o_template[1]  

# Sort ascending in wavelength (required by splrep)
sort_idx    = np.argsort(ccf_model_w)
ccf_model_w = ccf_model_w[sort_idx]
ccf_model_f = ccf_model_f[sort_idx]

print(f'CO template: {len(ccf_model_w)} points, '
      f'wavelength {ccf_model_w.min()*1e9:.1f} - {ccf_model_w.max()*1e9:.1f} nm')

# ---- RV lag grid: -100 to +100 km/s, step 1 km/s ----
RVlag = np.arange(-100, 101, 1)   # 201 points
ncc   = len(RVlag)

# ---- Prepare the data cube ----
# extracted_spectra_combined shape: (nDet=3, nOrder=5, nPix=2048)
# Add singleton spatial axis -> (nDet, nOrder, n_spatial=1, nPix)
nDet, nOrder, nPix = extracted_spectra_combined.shape
cube_combined = extracted_spectra_combined[:, :, np.newaxis, :]  # (3, 5, 1, 2048)

print(f'Obs cube shape  : {cube_combined.shape}')
print(f'Wave grid shape : {wave_data.shape}')
print(f'RV lag          : {RVlag[0]} to {RVlag[-1]} km/s  ({ncc} steps)')

# ---- Instantiate cross-correlator ----
process_cc_combined = processor_cross_correlation(
    wMod      = ccf_model_w * 1e9,   # metres -> nm to match wave_data
    fMod      = ccf_model_f,
    wlen      = wave_data,            # (nDet, nOrder, nPix) in nm
    cube      = cube_combined,        # (nDet, nOrder, 1, nPix)
    nOrder    = nOrder,
    nDet      = nDet,
    n_spatial = 1
)

# ---- Run cross-correlation ----
ccf_sum, ccf_snr = process_cc_combined.ccf_tot(
    rvlag         = RVlag,
    ncc           = ncc,
    plot          = False,
    normalization = 'median subtracted',
    clean_grids = [(0, 50), (150, 200)]
)

# ccf_sum / ccf_snr shape (1, 201) -> squeeze to 1D
ccf_1d     = ccf_sum[0, :]
ccf_snr_1d = ccf_snr[0, :]

peak_rv  = RVlag[np.nanargmax(ccf_snr_1d)]
peak_snr = np.nanmax(ccf_snr_1d)
print(f'\nPeak CCF SNR = {peak_snr:.2f}  at  RV = {peak_rv:.1f} km/s')

#---- Run auto corss-correlation -----

#using spline interpolation and reshape the model flux to (nDet, nOrder, 1, nPix) to match the cube shape
ccf_model_f_int = np.zeros((nDet, nOrder, nPix))
valid = np.isfinite(ccf_model_w) & np.isfinite(ccf_model_f)
coef_spline = interpolate.splrep(ccf_model_w[valid] * 1e9, ccf_model_f[valid], s=0.0)  # spline coefficients for the model
for iDet in range(nDet):
    for iOrder in range(nOrder):
        ccf_model_f_int[iDet, iOrder, :] = interpolate.splev(wave_data[iDet, iOrder, :], coef_spline, der=0)
        


ccf_model_f_reshaped = ccf_model_f_int[:, :, np.newaxis, :]  # (nDet, nOrder, 1, nPix)

process_cc_auto = processor_cross_correlation(
    wMod      = ccf_model_w * 1e9,   # metres -> nm to match wave_data
    fMod      = ccf_model_f,
    wlen      = wave_data,            # (nDet, nOrder, nPix) in nm
    cube      = ccf_model_f_reshaped,                  # (nDet, nOrder, 1, nPix)
    nOrder    = nOrder,
    nDet      = nDet,
    n_spatial = 1
)

ccf_auto, ccf_auto_snr = process_cc_auto.ccf_tot(
    rvlag         = RVlag,  
    ncc           = ncc,
    plot          = False,
    normalization = 'median subtracted',
    clean_grids = [(0, 50), (150, 200)]

)

ccf_auto_1d = ccf_auto[0, :]
ccf_auto_snr_1d = ccf_auto_snr[0, :]

peak_rv_auto = RVlag[np.nanargmax(ccf_auto_snr_1d)]
peak_snr_auto = np.nanmax(ccf_auto_snr_1d)
print(f'\nAuto CCF Peak SNR = {peak_snr_auto:.2f}  at  RV = {peak_rv_auto:.1f} km/s')

# ---- Plot results ----
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(RVlag, ccf_1d, color='steelblue', linewidth=1.5, label='H2O CCF')
axes[0].axvline(0,       color='gray',   linestyle='--', linewidth=0.8, label='RV = 0')
axes[0].axvline(peak_rv, color='tomato', linestyle='--', linewidth=1.0,
                label=f'Peak at {peak_rv:.0f} km/s')
#axes[0].plot(RVlag, ccf_auto_1d, color='purple', linewidth=1.5, label='Auto CCF')



axes[0].set_ylabel('CCF (median-subtracted)')
axes[0].set_title('H2O Cross-Correlation - Combined Spectra\n'
                  '(template: T=2300 K, log g=4.0, [M/H]=0.0, H2O only)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(RVlag, ccf_snr_1d, color='darkorange', linewidth=1.5, label='H2O CCF SNR')
axes[1].axvline(0,       color='gray',   linestyle='--', linewidth=0.8)
axes[1].axvline(peak_rv, color='tomato', linestyle='--', linewidth=1.0,
                label=f'Peak SNR = {peak_snr:.2f}')
axes[1].set_xlabel('RV lag (km/s)')
axes[1].set_ylabel('SNR')
axes[1].set_title('H2O CCF SNR')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(workpath + night + '/h2o_ccf_combined_spectra.png', dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
from scipy.interpolate import splrep, splev

#Follow the same CC workflow for CH4

#----- Load H2O template spectrum ----
h2o_template_path = workpath + '/spectra_input/planet_2300_4.0_-0.0_0_ch4_only.dat'
h2o_template = np.loadtxt(h2o_template_path)
ccf_model_w = h2o_template[0]
ccf_model_f = h2o_template[1]  

# Sort ascending in wavelength (required by splrep)
sort_idx    = np.argsort(ccf_model_w)
ccf_model_w = ccf_model_w[sort_idx]
ccf_model_f = ccf_model_f[sort_idx]

print(f'CO template: {len(ccf_model_w)} points, '
      f'wavelength {ccf_model_w.min()*1e9:.1f} - {ccf_model_w.max()*1e9:.1f} nm')

# ---- RV lag grid: -100 to +100 km/s, step 1 km/s ----
RVlag = np.arange(-100, 101, 1)   # 201 points
ncc   = len(RVlag)

# ---- Prepare the data cube ----
# extracted_spectra_combined shape: (nDet=3, nOrder=5, nPix=2048)
# Add singleton spatial axis -> (nDet, nOrder, n_spatial=1, nPix)
nDet, nOrder, nPix = extracted_spectra_combined.shape
cube_combined = extracted_spectra_combined[:, :, np.newaxis, :]  # (3, 5, 1, 2048)

print(f'Obs cube shape  : {cube_combined.shape}')
print(f'Wave grid shape : {wave_data.shape}')
print(f'RV lag          : {RVlag[0]} to {RVlag[-1]} km/s  ({ncc} steps)')

# ---- Instantiate cross-correlator ----
process_cc_combined = processor_cross_correlation(
    wMod      = ccf_model_w * 1e9,   # metres -> nm to match wave_data
    fMod      = ccf_model_f,
    wlen      = wave_data,            # (nDet, nOrder, nPix) in nm
    cube      = cube_combined,        # (nDet, nOrder, 1, nPix)
    nOrder    = nOrder,
    nDet      = nDet,
    n_spatial = 1
)

# ---- Run cross-correlation ----
ccf_sum, ccf_snr = process_cc_combined.ccf_tot(
    rvlag         = RVlag,
    ncc           = ncc,
    plot          = False,
    normalization = 'median subtracted',
    clean_grids = [(0, 50), (150, 200)]
)

# ccf_sum / ccf_snr shape (1, 201) -> squeeze to 1D
ccf_1d     = ccf_sum[0, :]
ccf_snr_1d = ccf_snr[0, :]

peak_rv  = RVlag[np.nanargmax(ccf_snr_1d)]
peak_snr = np.nanmax(ccf_snr_1d)
print(f'\nPeak CCF SNR = {peak_snr:.2f}  at  RV = {peak_rv:.1f} km/s')


#---- Run auto corss-correlation -----

#using spline interpolation and reshape the model flux to (nDet, nOrder, 1, nPix) to match the cube shape
ccf_model_f_int = np.zeros((nDet, nOrder, nPix))
coef_spline = interpolate.splrep(ccf_model_w * 1e9, ccf_model_f, s=0.0)  # spline coefficients for the model
splev = interpolate.splev  # ensure local name used below is defined
for iDet in range(nDet):
    for iOrder in range(nOrder):
        ccf_model_f_int[iDet, iOrder, :] = interpolate.splev(wave_data[iDet, iOrder, :], coef_spline, der=0)
        


ccf_model_f_reshaped = ccf_model_f_int[:, :, np.newaxis, :]  # (nDet, nOrder, 1, nPix)

process_cc_auto = processor_cross_correlation(
    wMod      = ccf_model_w * 1e9,   # metres -> nm to match wave_data
    fMod      = ccf_model_f,
    wlen      = wave_data,            # (nDet, nOrder, nPix) in nm
    cube      = ccf_model_f_reshaped,                  # (nDet, nOrder, 1, nPix)
    nOrder    = nOrder,
    nDet      = nDet,
    n_spatial = 1
)

ccf_auto, ccf_auto_snr = process_cc_auto.ccf_tot(
    rvlag         = RVlag,  
    ncc           = ncc,
    plot          = False,
    normalization = 'median subtracted',
    clean_grids = [(0, 50), (150, 200)]

)

ccf_auto_1d = ccf_auto[0, :]
ccf_auto_snr_1d = ccf_auto_snr[0, :]

peak_rv_auto = RVlag[np.nanargmax(ccf_auto_snr_1d)]
peak_snr_auto = np.nanmax(ccf_auto_snr_1d)
print(f'\nAuto CCF Peak SNR = {peak_snr_auto:.2f}  at  RV = {peak_rv_auto:.1f} km/s')

# ---- Plot results ----
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(RVlag, ccf_1d, color='steelblue', linewidth=1.5, label='CH4 CCF')
axes[0].axvline(0,       color='gray',   linestyle='--', linewidth=0.8, label='RV = 0')
axes[0].axvline(peak_rv, color='tomato', linestyle='--', linewidth=1.0,
                label=f'Peak at {peak_rv:.0f} km/s')

#axes[0].plot(RVlag, ccf_auto_1d, color='seagreen', linewidth=1.5, label='Auto CCF')



axes[0].set_ylabel('CCF (median-subtracted)')
axes[0].set_title('CH4 Cross-Correlation - Combined Spectra\n'
                  '(template: T=2300 K, log g=4.0, [M/H]=0.0, CH4 only)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(RVlag, ccf_snr_1d, color='darkorange', linewidth=1.5, label='CH4 CCF SNR')
axes[1].axvline(0,       color='gray',   linestyle='--', linewidth=0.8)
axes[1].axvline(peak_rv, color='tomato', linestyle='--', linewidth=1.0,
                label=f'Peak SNR = {peak_snr:.2f}')
axes[1].set_xlabel('RV lag (km/s)')
axes[1].set_ylabel('SNR')
axes[1].set_title('CH4 CCF SNR')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(workpath + night + '/ch4_ccf_combined_spectra.png', dpi=150, bbox_inches='tight')

plt.show()
